## Imports

In [1]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
print("Ready.")

Ready.


## Load the cleaned data

In [3]:
df = pd.read_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_clean.csv")
print(f"Loaded {len(df)} rows")
df.head()

Loaded 10000 rows


,reviewId,userName,review_text,rating,thumbs_up,review_date,reviewCreatedVersion,appVersion,replyContent,year,month,review_length,got_reply
0,614974df-eef0-4b87-887c-0d52ea3b4b12,Martyn Boyd,does exactly what I need it to do with more st...,5,0,2026-07-19 02:24:39,10.138,10.138,NaN,2026,7,60,False
1,ce70ff3b-a835-4f82-898f-5fdf3749adb6,Edward Krosendijk,nice,5,0,2026-07-18 23:49:34,10.138,10.138,NaN,2026,7,4,False
2,75a04d4f-9d10-4961-be06-6cca1e87302d,Steven Carey,Does what it says on the tin.,5,0,2026-07-18 23:43:32,10.138,10.138,NaN,2026,7,29,False
3,33a38df1-6cc8-4b87-bd2e-c2a83c4af526,Matt Kelly,Good app but frustrating that I can't see my r...,5,0,2026-07-18 22:41:09,10.138,10.138,Hi there. Thank you for your feedback. We're s...,2026,7,340,True
4,3faba9bd-8588-4885-92b2-c96552e37fc3,Ashu Kalonia,great,5,0,2026-07-18 22:27:19,10.138,10.138,NaN,2026,7,5,False


## VADER sentiment scoring

In [4]:
def get_sentiment_score(text):
    return analyzer.polarity_scores(str(text))["compound"]

df["sentiment_score"] = df["review_text"].apply(get_sentiment_score)
df[["review_text", "rating", "sentiment_score"]].head(10)

,review_text,rating,sentiment_score
0,does exactly what I need it to do with more st...,5,0.0000
1,nice,5,0.4215
2,Does what it says on the tin.,5,0.0000
3,Good app but frustrating that I can't see my r...,5,0.8151
4,great,5,0.6249
5,I like it,5,0.3612
6,it's working well,5,0.2732
7,"Stopped supporting older systems, spams notifi...",1,-0.7531
8,good tres bien,5,0.4404
9,solid card can have multiple currencies simult...,4,0.1531


## Convert score into a label

In [5]:
def score_to_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df["sentiment_label"] = df["sentiment_score"].apply(score_to_label)
df["sentiment_label"].value_counts()

sentiment_label
positive    7666
neutral     1233
negative    1101
Name: count, dtype: int64

##  Sanity check: does sentiment agree with star rating?

In [6]:
pd.crosstab(df["rating"], df["sentiment_label"])

sentiment_label,negative,neutral,positive
rating,,,
1,753,256,334
2,66,35,80
3,49,36,99
4,69,82,558
5,164,824,6595


## Define themes with keywords

In [9]:
themes = {
    "account_freeze": ["freeze", "frozen", "block", "blocked", "locked", "lock"],
    "customer_support": ["support", "customer service", "help desk", "chatbot", "agent", "response time"],
    "fees": ["fee", "charge", "charged", "expensive", "cost", "commission"],
    "app_stability": ["bug", "crash", "glitch", "freezing app", "not working", "won't open", "keeps closing"],
    "app_ux_positive": ["easy to use", "user friendly", "interface", "design", "simple", "intuitive"],
    "transfers": ["transfer", "payment", "send money", "withdraw", "deposit"],
    "verification": ["verify", "verification", "kyc", "id check", "document"],
}

def tag_themes(text):
    text = str(text).lower()
    matched = [theme for theme, keywords in themes.items() if any(kw in text for kw in keywords)]
    return matched if matched else ["other"]

df["themes"] = df["review_text"].apply(tag_themes)
df[["review_text", "themes"]].head(10)

,review_text,themes
0,does exactly what I need it to do with more st...,[other]
1,nice,[other]
2,Does what it says on the tin.,[other]
3,Good app but frustrating that I can't see my r...,[other]
4,great,[other]
5,I like it,[other]
6,it's working well,[other]
7,"Stopped supporting older systems, spams notifi...","[customer_support, app_ux_positive]"
8,good tres bien,[other]
9,solid card can have multiple currencies simult...,[other]


## Theme Analysis

### Methodology

Each review was scanned for keywords associated with six recurring topics in Revolut's customer feedback:

- **account_freeze** — account or card suspension/restriction issues
- **customer_support** — quality and responsiveness of customer service
- **fees** — charges, commissions, or perceived cost complaints
- **app_stability** — bugs, crashes, or technical malfunctions
- **app_ux_positive** — praise for ease of use and interface design
- **transfers** — issues or praise related to sending/receiving money
- **verification** — KYC/identity verification process complaints

Reviews matching none of these keywords are tagged `other`. A single review can match multiple themes if it mentions more than one topic.

**Limitation:** this is a keyword-based approach, not a trained classifier — it will miss complaints phrased in ways not captured by the keyword list, and may occasionally mis-tag reviews where a keyword appears in an unrelated context. Results should be read as directional, not exact.


##  Explode themes for counting

In [13]:
df_themes = df.explode("themes")
df_themes = df_themes.reset_index(drop=True)   # <-- fixes duplicate index issue

theme_counts = df_themes["themes"].value_counts()
theme_counts

themes
other               7552
app_ux_positive     1065
transfers            683
customer_support     415
fees                 388
verification         178
app_stability        167
account_freeze       164
Name: count, dtype: int64

## Theme breakdown by sentiment

In [14]:
theme_sentiment = pd.crosstab(df_themes["themes"], df_themes["sentiment_label"])
theme_sentiment

sentiment_label,negative,neutral,positive
themes,,,
account_freeze,82,12,70
app_stability,53,53,61
app_ux_positive,47,29,989
customer_support,181,14,220
fees,112,25,251
other,585,1046,5921
transfers,142,45,496
verification,94,29,55


## Focus on 1-star reviews

In [15]:
one_star = df_themes[df_themes["rating"] == 1]
one_star["themes"].value_counts()

themes
other               719
customer_support    238
transfers           152
verification        129
fees                116
account_freeze      113
app_stability        83
app_ux_positive      26
Name: count, dtype: int64

##  Save the enriched dataset

In [16]:
df.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_analyzed.csv", index=False)
print("Saved to E:/project/Revolut-analysis-deashboard/Data ")

Saved to E:/project/Revolut-analysis-deashboard/Data 
